# Ordered Logistic Regression Results Dataset Exploration with `mlcroissant`
This notebook demonstrates how to load, explore, and process the FAIR^2 dataset using the `mlcroissant` library.

### Dataset Source
The dataset's Croissant schema is publicly accessible at:

`https://sen.science/doi/10.71728/senscience.y7m0-f273/fair2.json`

In [ ]:
# Ensure `mlcroissant` is installed (run once)
!pip install -q mlcroissant

## 1. Data Loading
Load the dataset metadata and its records using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Dataset Croissant schema URL
croissant_url = "https://sen.science/doi/10.71728/senscience.y7m0-f273/fair2.json"

# Load dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata  # This is a metadata object (not a dict)

print(f"Dataset loaded: {metadata.name}")
print("Description:")
print(metadata.description)


## 2. Data Overview
List the available record sets and their fields, referencing all items by their `@id` values.

> **Note:** All Croissant schema entities (record sets, fields, columns) are identified by their `@id`, which ensures consistent referencing.

In [ ]:
# Explore the record sets defined in Croissant metadata.

record_sets = list(dataset.record_sets)
print(f"Found {len(record_sets)} record set(s):\n")

for rs in record_sets:
    print(f"- Name: {getattr(rs, 'name', None)}")
    print(f"  @id:  {rs.id}")
    if hasattr(rs, 'fields'):
        print(f"  Fields (by @id):")
        for fld in rs.fields:
            print(f"    - {fld.id}")
    print()

## 3. Data Extraction
Load records from a specific record set into a DataFrame for further processing.

We'll select the main record set (typically the core data, e.g., survey results) and use its `@id`. If the dataset contains multiple record sets, you can extract data from any/all of them, identified via their `@id`.

In [ ]:
# Load all record sets into DataFrames, using their @id as keys.

dataframes = {}
for rs in record_sets:
    records = list(dataset.records(record_set=rs.id))
    dataframes[rs.id] = pd.DataFrame(records)
    if not dataframes[rs.id].empty:
        print(f"\nRecord set {rs.id} loaded. Columns:")
        print(dataframes[rs.id].columns.tolist())
        print(dataframes[rs.id].head(2))
    else:
        print(f"\nRecord set {rs.id} is empty or contains no records.")


## 4. Exploratory Data Analysis (EDA)
Perform initial filtering, normalization, and grouping with data extracted from the selected record set.

> For illustration: Let's assume the main numeric field in the data is named by its `@id` (e.g., for a field containing log likelihood values, or similar regression outputs).

Replace `<main_record_set_id>` and `<main_numeric_field_id>` below with actual values from step 2/3. Common choices might be regression model results or survey data tables.

_If there are no numeric fields, adjust to use a relevant one for grouping/filtering._

In [ ]:
# Select a record set with data for EDA
main_rs = None
for rs in record_sets:
    if not dataframes[rs.id].empty:
        main_rs = rs
        break

assert main_rs is not None, "No non-empty record set found. Cannot proceed."

# Infer a numeric field from the columns (e.g., one with float or int values)
df = dataframes[main_rs.id]
numeric_field_id = None
for col in df.columns:
    if pd.api.types.is_numeric_dtype(df[col]):
        numeric_field_id = col
        break
if numeric_field_id is None:
    raise ValueError("No numeric field found for EDA in the main record set.")

print(f"Using record set @id: {main_rs.id}")
print(f"Using numeric field @id: {numeric_field_id}")

threshold = df[numeric_field_id].mean()
filtered_df = df[df[numeric_field_id] > threshold]
print(f"Filtered records with {numeric_field_id} > {threshold:.2f}:")
print(filtered_df.head())

# Normalize the numeric field for filtered records
filtered_df[f"{numeric_field_id}_normalized"] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
print(f"\nNormalized {numeric_field_id} for filtered records:")
print(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

# Attempt grouping by a categorical field (pick a non-numeric column)
group_field_id = None
for col in df.columns:
    if not pd.api.types.is_numeric_dtype(df[col]):
        group_field_id = col
        break
if group_field_id:
    grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean().reset_index()
    print(f"\nGrouped data by {group_field_id} (mean of {numeric_field_id}):")
    print(grouped_df.head())
else:
    print("No suitable categorical field found for grouping.")

## 5. Visualization
Visualize the distribution of the selected numeric field and its relationship to a categorical grouping (if available).


In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

plt.figure(figsize=(7, 4))
sns.histplot(df[numeric_field_id], kde=True, color='steelblue')
plt.title(f"Distribution of {numeric_field_id} in record set {main_rs.id}")
plt.xlabel(numeric_field_id)
plt.show()

# If grouping field found, show a boxplot
if group_field_id:
    plt.figure(figsize=(8, 5))
    sns.boxplot(x=group_field_id, y=numeric_field_id, data=df)
    plt.title(f"{numeric_field_id} by {group_field_id}")
    plt.grid(True, axis='y')
    plt.show()

## 6. Conclusion
- This notebook loaded and explored the FAIR^2 dataset using the `mlcroissant` library via its Croissant JSON-LD schema.
- All data elements were referenced by their `@id` attributes to ensure reproducibility and schema-consistent access.
- Basic EDA demonstrated numeric filtering, normalization, grouping, and visualization of key dataset variables.
- This approach can be extended to deeper analysis, modeling, or cross-referencing with other Croissant datasets.